[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [LightGBM and XGBoost, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)

# The Native Interfaces &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import mean_squared_error

TERRAINS = ["coastal", "fjord", "inland", "valley", "mountain"]


def station_days(n_stations=400, days=50, seed=2026):
    """Made-up daily minimum temperatures at weather stations in Norway, one row per station and day."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n_stations)
    coast_km = np.minimum(rng.exponential(30, n_stations), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n_stations) + 2.5 * coast_km, 0, 1600)
    terrain = rng.choice(TERRAINS, n_stations, p=[0.2, 0.2, 0.25, 0.2, 0.15])
    has_snow_sensor = rng.uniform(size=n_stations) < 0.7
    offset = rng.normal(0, 0.8, n_stations)
    station = np.repeat(np.arange(n_stations), days)
    n = n_stations * days
    day = rng.integers(1, 366, n)
    cloud = np.clip(rng.beta(1.2, 1.0, n), 0, 1)
    wind = rng.gamma(2.0, 2.0, n)
    season = np.cos(2 * np.pi * (day - 20) / 365)
    lat, elev, coast = latitude[station], elevation[station], coast_km[station]
    snow = np.where(season > 0.2, np.clip(rng.normal(30 * season + 0.03 * elev + 2 * (lat - 58), 15), 0, None), 0.0)
    shelter = pd.Series(terrain[station]).map({"coastal": 0.4, "fjord": 1.0, "inland": 1.2, "valley": 1.7, "mountain": 0.8}).to_numpy()
    mean = 8 - 0.55 * (lat - 58) - 0.006 * elev - 9 * season + 3.5 * np.exp(-coast / 40) * season
    cooling = 7 * (1 - cloud) * np.exp(-wind / 3) * (0.4 + (season + 1) / 2) * shelter
    tmin = mean - 2.5 - cooling - 2.0 * (snow > 10) * (1 - cloud) + offset[station] + rng.normal(0, 1.2, n)
    snow_cm = np.where(has_snow_sensor[station], snow.round(0), np.nan)
    return pd.DataFrame({
        "station": [f"ST{s + 1:04d}" for s in station],
        "latitude": lat.round(2), "elevation_m": elev.round(0).astype(int), "coast_km": coast.round(1),
        "terrain": pd.Categorical(terrain[station], categories=TERRAINS),
        "day_of_year": day, "cloud_cover": cloud.round(2), "wind_ms": wind.round(1), "snow_cm": snow_cm,
        "tmin": tmin.round(1),
    })


FEATURES = ["latitude", "elevation_m", "coast_km", "terrain", "day_of_year", "cloud_cover", "wind_ms", "snow_cm"]
days = station_days()
stations = days["station"].unique()
order = np.random.default_rng(0).permutation(len(stations))
train = days[days["station"].isin(stations[order[:300]])]
valid = days[days["station"].isin(stations[order[300:350]])]
test = days[days["station"].isin(stations[order[350:]])]

print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__, "|", len(train), "training rows,",
      len(valid), "validation rows,", len(test), "test rows")

lightgbm 4.6.0 | xgboost 3.4.1 | 15000 training rows, 2500 validation rows, 2500 test rows


The worked examples' tables and data objects, rebuilt here from the same made-up stations so that
each answer runs on its own.

In [2]:
y_train, y_valid = train["tmin"], valid["tmin"]
train_dataset = lgb.Dataset(train[FEATURES], label=y_train)
valid_dataset = lgb.Dataset(valid[FEATURES], label=y_valid, reference=train_dataset)
dtrain = xgb.DMatrix(train[FEATURES], label=y_train)
dvalid = xgb.DMatrix(valid[FEATURES], label=y_valid)

**1.** Train a LightGBM booster with `lgb.train`, `num_leaves` of 15 and a learning rate of 0.05, for 300 rounds, and print its validation loss.

In [3]:
params = {"objective": "regression", "num_leaves": 15, "learning_rate": 0.05, "seed": 0, "verbose": -1}
booster = lgb.train(params, train_dataset, num_boost_round=300)
print(round(mean_squared_error(y_valid, booster.predict(valid[FEATURES])), 3))

2.516


15 leaves at a learning rate of 0.05 for 300 rounds give a validation loss of 2.516, against the 2.553 that the worked examples recorded for LightGBM's defaults, 31 leaves at 0.1, at round 100, a difference that one run is too little to rank.

**2.** Train an XGBoost booster with `xgb.train`, a `max_depth` of 4 and a learning rate of 0.1, for 300 rounds, and print its validation loss.

In [4]:
params = {"objective": "reg:squarederror", "max_depth": 4, "learning_rate": 0.1, "seed": 0}
booster = xgb.train(params, dtrain, num_boost_round=300)
print(round(mean_squared_error(y_valid, booster.predict(dvalid)), 3))

2.474


A depth of 4 at 0.1 for 300 rounds gives a validation loss of 2.474, against the 2.924 that the worked examples recorded for XGBoost's defaults, a depth of 6 at 0.3, at round 100.

**3.** Record LightGBM's validation loss for 300 rounds at a learning rate of 0.1, as the worked examples do, and print the round with the lowest loss and that loss.

In [5]:
params = {"objective": "regression", "learning_rate": 0.1, "num_leaves": 31, "seed": 0, "verbose": -1}
losses = {}
lgb.train(params, train_dataset, num_boost_round=300, valid_sets=[valid_dataset], valid_names=["valid"],
          callbacks=[lgb.record_evaluation(losses)])
l2 = losses["valid"]["l2"]
best = int(np.argmin(l2))
print("round", best + 1, "loss", round(l2[best], 3))

round 120 loss 2.546


The validation loss is lowest at round 120, 2.546. `np.argmin` counts from 0, so the round is one more. The **Learning Rate and Trees** notebook reads this curve for several learning rates.

**4.** Fit `LGBMRegressor(n_estimators=200, num_leaves=15, random_state=0, verbose=-1)`, and print the number of leaves, the learning rate and the number of rounds in its `booster_.params`, and the number of trees the booster has.

In [6]:
model = lgb.LGBMRegressor(n_estimators=200, num_leaves=15, random_state=0, verbose=-1).fit(train[FEATURES], y_train)
used = model.booster_.params
print({setting: used[setting] for setting in ("num_leaves", "learning_rate", "num_iterations")})
print(model.booster_.num_trees())

{'num_leaves': 15, 'learning_rate': 0.1, 'num_iterations': 200}
200


The scikit-learn model passed its settings to the booster, the number of rounds under LightGBM's own name, `num_iterations`, and the booster trained 200 trees.

**5.** Give `xgb.train` both the training and the validation `DMatrix` in `evals`, and print the last round's `rmse` for each.

In [7]:
params = {"objective": "reg:squarederror", "learning_rate": 0.3, "max_depth": 6, "seed": 0}
losses = {}
xgb.train(params, dtrain, num_boost_round=100, evals=[(dtrain, "train"), (dvalid, "valid")], evals_result=losses,
          verbose_eval=False)
print({name: round(values["rmse"][-1], 3) for name, values in losses.items()})

{'train': 0.958, 'valid': 1.71}


After 100 rounds the training stations' rmse is 0.958 and the validation stations' 1.71, so the model fits the stations it trained on far more closely than new ones.

---

&#8592; **Back to:** [The Native Interfaces](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/lightgbm-and-xgboost-deep-dive/03-the-native-interfaces.ipynb)  &nbsp;&middot;&nbsp;  [LightGBM and XGBoost, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/lightgbm-and-xgboost-deep-dive.html)
